In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import copy
from pathlib import Path
import pandas as pd
import random
import re
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import sklearn.metrics
from transformers import RobertaModel
from transformers import RobertaConfig
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from torch.autograd import Variable
from torch import nn, optim
from torch.optim import SGD,Adam,RMSprop
from torch.utils.data import Dataset, DataLoader, IterableDataset
from clang import *

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/VulnSight")

In [ ]:
seed = 1234
os.environ['PYTHONHASHSEED'] = str(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.enabled = True
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
multigpu = False
if device == torch.device('cuda'):
	multigpu = torch.cuda.device_count() > 1
print('Device: ',device)
print('MultiGPU: ',multigpu)

Device:  cuda
MultiGPU:  False


In [ ]:
## Training & vocab parameters
VOCAB_SIZE = 50000
BATCH_SIZE = 32   # 128
EMBED_SIZE = VOCAB_SIZE+2
EMBED_DIM = 768

In [ ]:
## Tokenizer

from tokenizers.pre_tokenizers import PreTokenizer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers import NormalizedString,PreTokenizedString
from typing import List

class MyTokenizer:

    cidx = cindex.Index.create()


    def clang_split(self, i: int, normalized_string: NormalizedString) -> List[NormalizedString]:
        ## Tokenize using clang
        tok = []
        tu = self.cidx.parse('tmp.c',
                       args=[''],
                       unsaved_files=[('tmp.c', str(normalized_string.original))],
                       options=0)
        for t in tu.get_tokens(extent=tu.cursor.extent):
            spelling = t.spelling.strip()

            if spelling == '':
                continue

            tok.append(NormalizedString(spelling))

        return(tok)

    def pre_tokenize(self, pretok: PreTokenizedString):
        pretok.split(self.clang_split)


In [ ]:
## Custom tokenizer

from tokenizers import Tokenizer
from tokenizers import normalizers,decoders
from tokenizers.normalizers import StripAccents, unicode_normalizer_from_str, Replace
from tokenizers.processors import TemplateProcessing
from tokenizers import processors,pre_tokenizers
from tokenizers.models import BPE

## Load pre-trained tokenizers
vocab_path = str(PROJECT_ROOT / "tokenizer" / "drapgh-vocab.json")
merge_path = str(PROJECT_ROOT / "tokenizer" / "drapgh-merges.txt")

vocab, merges = BPE.read_file(vocab=vocab_path, merges=merge_path)
my_tokenizer = Tokenizer(BPE(vocab, merges, unk_token="<unk>"))

my_tokenizer.normalizer = normalizers.Sequence([StripAccents(), Replace(" ", "Ä")])
my_tokenizer.pre_tokenizer = PreTokenizer.custom(MyTokenizer())
my_tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    special_tokens=[
    ("<s>",0),
    ("<pad>",1),
    ("</s>",2),
    ("<unk>",3),
    ("<mask>",4)
    ]
)

In [ ]:
mydataset = 'draper'

In [ ]:
my_tokenizer.enable_truncation(max_length=1024)
my_tokenizer.enable_padding(direction='right', pad_id=1, pad_type_id=0, pad_token='<pad>', length=None, pad_to_multiple_of=None)

In [ ]:
def cleaner(code):
    ## Remove code comments
    pat = re.compile(r'(/\*([^*]|(\*+[^*/]))*\*+/)|(//.*)')
    code = re.sub(pat,'',code)
    code = re.sub('\n','',code)
    code = re.sub('\t','',code)
    return(code)

def process_encodings(encodings):
    input_ids=[]
    attention_mask=[]
    for enc in encodings:
        input_ids.append(enc.ids)
        attention_mask.append(enc.attention_mask)
    return {'input_ids':input_ids, 'attention_mask':attention_mask}

def replace_colname(x):
    x = x.rename(columns={
        'functionSource': 'func',
        'code': 'func',
        'label': 'target'
    })
    return x

In [ ]:
if mydataset == 'devign':
    devign_dir = PROJECT_ROOT / "finetune" / "devign"
    train_index_path = devign_dir / "train.txt"
    valid_index_path = devign_dir / "valid.txt"
    devign_json_path = devign_dir / "Devign.json"

    for p in (train_index_path, valid_index_path, devign_json_path):
        if not p.exists():
            raise FileNotFoundError(f"Required Devign file not found: {p}")

    train_index = set()
    valid_index = set()

    with open(train_index_path) as f:
        for line in f:
            line = line.strip()
            if line:
                train_index.add(int(line))

    with open(valid_index_path) as f:
        for line in f:
            line = line.strip()
            if line:
                valid_index.add(int(line))

    mydata = pd.read_json(devign_json_path)
    m1 = mydata.iloc[list(train_index)].copy()
    m2 = mydata.iloc[list(valid_index)].copy()
    del mydata

elif mydataset == 'd2a':
    task = 'function'
    d2a_dir = PROJECT_ROOT / "finetune" / "d2a" / task
    train_path = d2a_dir / f"d2a_lbv1_{task}_train.csv"
    val_path = d2a_dir / f"d2a_lbv1_{task}_dev.csv"

    for p in (train_path, val_path):
        if not p.exists():
            raise FileNotFoundError(
                f"Required D2A file not found: {p}\n"
                "Update d2a_dir/train_path/val_path to match your Google Drive folders."
            )

    m1 = pd.read_csv(train_path)
    m2 = pd.read_csv(val_path)
    m1 = replace_colname(m1)
    m2 = replace_colname(m2)

else:
    dataset_dir = PROJECT_ROOT / "finetune" / mydataset.lower()

    train_path = dataset_dir / f"{mydataset.lower()}_train.pkl"
    val_path = dataset_dir / f"{mydataset.lower()}_val.pkl"

    for p in (train_path, val_path):
        if not p.exists():
            raise FileNotFoundError(
                f"Required dataset file not found: {p}\n"
                "Update train_path and val_path to match your Google Drive folders."
            )

    m1 = pd.read_pickle(train_path)
    m2 = pd.read_pickle(val_path)
    m1 = replace_colname(m1)
    m2 = replace_colname(m2)


# Clean and tokenize train and validation data
m1.func = m1.func.apply(cleaner)
train_encodings = my_tokenizer.encode_batch(m1.func)
try:
    train_encodings = [
        {'func': enc.ids, 'target': lab}
        for enc, lab in zip(train_encodings, m1.target.tolist())
    ]
except Exception:
    train_encodings = [
        {'func': enc.ids, 'target': lab}
        for enc, lab in zip(train_encodings, (m1['combine'] * 1).tolist())
    ]

m2.func = m2.func.apply(cleaner)
val_encodings = my_tokenizer.encode_batch(m2.func)
try:
    val_encodings = [
        {'func': enc.ids, 'target': lab}
        for enc, lab in zip(val_encodings, m2.target.tolist())
    ]
except Exception:
    val_encodings = [
        {'func': enc.ids, 'target': lab}
        for enc, lab in zip(val_encodings, (m2['combine'] * 1).tolist())
    ]


In [ ]:
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace

class VulSightDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        item = self.encodings[idx]

        return {
            "codes": torch.tensor(
                item["func"], dtype=torch.long
            ),
            "label": torch.tensor(
                int(item["target"]), dtype=torch.long
            )
        }


train_data = VulSightDataset(train_encodings)
val_data = VulSightDataset(val_encodings)


def collate_batch(batch):
    codes = torch.stack([item["codes"] for item in batch])
    labels = torch.stack([item["label"] for item in batch])

    # Preserve the attribute names expected by your training loop
    return SimpleNamespace(codes=codes, label=labels)


train_iterator = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

valid_iterator = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Training batches:", len(train_iterator))
print("Validation batches:", len(valid_iterator))

Training examples: 21854
Validation examples: 2732
Training batches: 683
Validation batches: 86


In [ ]:
# Verify one example
item = train_data[0]

codes = item["codes"]
label = item["label"]

print("Code shape:", codes.shape)
print("Label:", label)
print("Code dtype:", codes.dtype)
print("Label dtype:", label.dtype)

Code shape: torch.Size([1024])
Label: tensor(0)
Code dtype: torch.int64
Label dtype: torch.int64


### Define VulSight-CNN model

In [ ]:
class myCNN(nn.Module):
    def __init__(self, EMBED_SIZE, EMBED_DIM):
        super(myCNN,self).__init__()

        pretrained_model_path = PROJECT_ROOT / "models" / "VulSight"
        if not pretrained_model_path.exists():
            raise FileNotFoundError(f"Pretrained VulSight model directory not found: {pretrained_model_path}")
        pretrained_weights = RobertaModel.from_pretrained(str(pretrained_model_path)).embeddings.word_embeddings.weight

        self.embed = nn.Embedding.from_pretrained(pretrained_weights,
                                                  freeze=True,
                                                  padding_idx=1)

        self.conv1 = nn.Conv1d(in_channels=EMBED_DIM, out_channels=200, kernel_size=3)
        self.conv2 = nn.Conv1d(in_channels=EMBED_DIM, out_channels=200, kernel_size=4)
        self.conv3 = nn.Conv1d(in_channels=EMBED_DIM, out_channels=200, kernel_size=5)

        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(200*3,256)
        self.fc2 = nn.Linear(256,128)
        self.fc3 = nn.Linear(128,2)

    def forward(self, x):
        x = self.embed(x)
        x = x.permute(0,2,1)

        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x))
        x3 = F.relu(self.conv3(x))

        x1 = F.max_pool1d(x1, x1.shape[2])
        x2 = F.max_pool1d(x2, x2.shape[2])
        x3 = F.max_pool1d(x3, x3.shape[2])

        x = torch.cat([x1,x2,x3],dim=1)

        # flatten the tensor
        x = x.flatten(1)

        # apply mean over the last dimension
        #x = torch.mean(x, -1)

        x = self.dropout(x)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return(x)

In [ ]:
model = myCNN(EMBED_SIZE,EMBED_DIM)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: /content/drive/MyDrive/VulnSight/models/VulSight
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print("Embedding shape:", tuple(model.embed.weight.shape))
print("CNN expected dimension:", EMBED_DIM)

assert model.embed.weight.shape[1] == EMBED_DIM

Embedding shape: (50000, 768)
CNN expected dimension: 768


In [ ]:
PAD_IDX = 1 #in vocab
model.embed.weight.data[PAD_IDX] = torch.zeros(EMBED_DIM)

In [ ]:
if multigpu:
    model = torch.nn.DataParallel(model)
model.to(device)
print(model)

myCNN(
  (embed): Embedding(50000, 768, padding_idx=1)
  (conv1): Conv1d(768, 200, kernel_size=(3,), stride=(1,))
  (conv2): Conv1d(768, 200, kernel_size=(4,), stride=(1,))
  (conv3): Conv1d(768, 200, kernel_size=(5,), stride=(1,))
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=600, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=2, bias=True)
)


In [ ]:
print('Num of trainable param: ',sum(p.numel() for p in model.parameters() if p.requires_grad))

Num of trainable param:  2030810


### Prepare loss function

In [ ]:
# Extract labels from the training DataFrame
y_train = m1["target"].to_numpy()

# Calculate balanced class weights
classes = np.unique(y_train)

cm = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

c_weights = torch.tensor(
    cm, dtype=torch.float32
).to(device)

criterion = torch.nn.CrossEntropyLoss(
    weight=c_weights
).to(device)

optimizer = Adam(model.parameters(), lr=0.0005)

In [ ]:
def softmax_accuracy(probs,all_labels):
    def getClass(x):
        return(x.index(max(x)))

    all_labels = all_labels.tolist()
    probs = pd.Series(probs.tolist())
    all_predicted = probs.apply(getClass)
    all_predicted.reset_index(drop=True, inplace=True)
    vc = pd.value_counts(all_predicted == all_labels)
    try:
        acc = vc[1]/len(all_labels)
    except:
        if(vc.index[0]==False):
            acc = 0
        else:
            acc = 1
    return(acc)

### Start training

In [ ]:
# Name and create the checkpoint directory in Google Drive.
model_foldername = f"VulSight-CNN_{mydataset}"
checkpoint_dir = PROJECT_ROOT / "models" / model_foldername
checkpoint_dir.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints will be saved to: {checkpoint_dir}")

Checkpoints will be saved to: /content/drive/MyDrive/VulnSight/models/VulSight-CNN_devign


In [ ]:
import time
import copy
import torch

print("Training started.....")

EPOCHS = 10
BEST_VAL = float("inf")
BEST_MODEL = None
BEST_EPOCH = None

# Ensure the checkpoint directory exists
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for e in range(EPOCHS):
    running_acc = 0.0
    running_loss = 0.0
    timer = time.time()

    # -------------------------
    # Training
    # -------------------------
    model.train()

    for batch in train_iterator:
        batch.codes = batch.codes.to(device)
        batch.label = batch.label.to(device)

        optimizer.zero_grad()

        output = model(batch.codes)
        loss = criterion(output, batch.label)

        loss.backward()
        optimizer.step()

        acc = softmax_accuracy(output, batch.label)

        running_acc += acc
        running_loss += loss.item()

    train_acc = running_acc / len(train_iterator)
    train_loss = running_loss / len(train_iterator)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    running_acc_val = 0.0
    running_loss_val = 0.0

    with torch.no_grad():
        for batch in valid_iterator:
            batch.codes = batch.codes.to(device)
            batch.label = batch.label.to(device)

            output_val = model(batch.codes)
            loss_val = criterion(output_val, batch.label)

            acc_val = softmax_accuracy(output_val, batch.label)

            running_acc_val += acc_val
            running_loss_val += loss_val.item()

    val_acc = running_acc_val / len(valid_iterator)
    val_loss = running_loss_val / len(valid_iterator)

    # -------------------------
    # Print epoch results
    # -------------------------
    elapsed_time = time.time() - timer

    print_out = (
        f"Epoch {e + 1} - "
        f"Training acc: {train_acc:.4f} - "
        f"Training loss: {train_loss:.4f} - "
        f"Val acc: {val_acc:.4f} - "
        f"Val loss: {val_loss:.4f} - "
        f"Time: {elapsed_time:.4f}s"
    )

    print(print_out)

    # -------------------------
    # Save every epoch checkpoint
    # -------------------------
    epoch_checkpoint_path = (
        checkpoint_dir / f"model_ep_{e + 1}.tar"
    )

    torch.save(
        {
            "epoch": e + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,
        },
        epoch_checkpoint_path,
    )

    # -------------------------
    # Track the best model
    # -------------------------
    if val_loss < BEST_VAL:
        BEST_VAL = val_loss
        BEST_EPOCH = e + 1

        # Store an independent copy of the best weights
        BEST_MODEL = copy.deepcopy(model.state_dict())

        print(f"Validation loss decreased! Best epoch: {BEST_EPOCH}")

        # Save the best checkpoint separately
        best_checkpoint_path = checkpoint_dir / "best_model.tar"

        torch.save(
            {
                "epoch": BEST_EPOCH,
                "model_state_dict": BEST_MODEL,
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": BEST_VAL,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_acc": val_acc,
            },
            best_checkpoint_path,
        )

    print("-" * 80)

print("Training completed!")

# -------------------------
# Restore the best model
# -------------------------
if BEST_MODEL is not None:
    model.load_state_dict(BEST_MODEL)

    print(f"Best epoch: {BEST_EPOCH}")
    print(f"Best validation loss: {BEST_VAL:.4f}")
    print("Best model weights restored successfully.")
else:
    print("No best model was saved. Check the training and validation data.")


Training started.....
Epoch 1 - Training acc: 0.4197 - Training loss: 0.6840 - Val acc: 0.4111 - Val loss: 0.6869 - Time: 68.3862s
Validation loss decreased! Best epoch: 1
--------------------------------------------------------------------------------
Epoch 2 - Training acc: 0.4046 - Training loss: 0.6649 - Val acc: 0.3904 - Val loss: 0.6480 - Time: 67.9281s
Validation loss decreased! Best epoch: 2
--------------------------------------------------------------------------------
Epoch 3 - Training acc: 0.3698 - Training loss: 0.6305 - Val acc: 0.3712 - Val loss: 0.6069 - Time: 67.1573s
Validation loss decreased! Best epoch: 3
--------------------------------------------------------------------------------
Epoch 4 - Training acc: 0.3416 - Training loss: 0.5944 - Val acc: 0.3388 - Val loss: 0.6035 - Time: 67.2173s
Validation loss decreased! Best epoch: 4
--------------------------------------------------------------------------------
Epoch 5 - Training acc: 0.3202 - Training loss: 0.5651